# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.

In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get a dictionary version of the metadata for pretty printing
metadata = dataset.metadata.to_json()

print(f"Dataset: {metadata.get('name', '<no name>')}")
print(f"Description: {metadata.get('description', '<no description>')}")

# Display key info
print(f"Published: {metadata.get('datePublished', '<date unknown>')}")
print(f"Keywords: {', '.join(metadata.get('keywords', []))}")


## 2. Data Overview
Review available record sets and their IDs as defined by the Croissant schema. Each record set, field, and column has a unique `@id`.

Below, we enumerate all record sets available in the dataset and their respective field (column) `@id`s.

In [ ]:
# Get record sets
record_sets = dataset.metadata.record_sets
print("Record sets found in the dataset:")
for rs in record_sets:
    print(f"  - Name: {rs.name}")
    print(f"    @id: {rs.id}")
    # List fields
    print(f"    Fields (columns):")
    for field in rs.fields:
        print(f"      - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load tabular data from the main record set for analysis.

The FAIR² dataset contains one main record set (typically containing MS protein/peptide tables). You can use its `@id` directly to extract records.

In [ ]:
# For demonstration, extract all record sets (likely just one, but notebook will work with more)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    # Each record is returned as a dict using Croissant field @id as keys
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Records loaded: {len(dataframes[rs_id])}, Columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"No records found for record set {rs_id}")
    print("")

# For the main record set, show the first 5 rows and columns
if record_set_ids:
    rs_main_id = record_set_ids[0]
    print(f"First 5 rows of record set: {rs_main_id}")
    display(dataframes[rs_main_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering records, normalizing numeric fields, removing outliers, and grouping.

Make sure to reference each field by its corresponding `@id`.

In [ ]:
# EDA on the main record set
if record_set_ids:
    main_df = dataframes[rs_main_id]
    main_record_set = [rs for rs in dataset.metadata.record_sets if rs.id == rs_main_id][0]

    # Find numeric fields (e.g. intensity, MW, peptide counts, etc.)
    numeric_field_ids = [field.id for field in main_record_set.fields if field.data_type in ['schema:Float', 'schema:Number', 'schema:Integer']]
    print(f"Numeric fields in main record set ({rs_main_id}):\n", json.dumps(numeric_field_ids, indent=2))
    
    # Pick a commonly useful numeric field, or the first available
    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        
        # Filter: keep only records where value is above a threshold (e.g., mean or fixed number)
        threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} remaining")
        display(filtered_df.head())

        # Normalize the numeric field (z-score normalization)
        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First normalized records ({numeric_field_id}):")
        display(filtered_df[[numeric_field_id, norm_field]].head())

        # Try grouping by a categorical field (e.g. "description", "accession", etc.)
        group_field_ids = [field.id for field in main_record_set.fields if field.data_type in ['schema:Text']]
        if group_field_ids:
            group_field = group_field_ids[0]
            print(f"Grouping results by {group_field} (showing mean {numeric_field_id}):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
            display(grouped_df.head())
        else:
            print("No text/categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets found for EDA.")

## 5. Visualization
Visualize distributions in the data, such as histograms of numeric fields, or boxplots grouped by categorical attributes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize for the main record set
if record_set_ids and numeric_field_ids:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot group by a categorical field if available
    if group_field_ids:
        plt.figure(figsize=(10,4))
        top_groups = main_df[group_field].value_counts().index[:5]
        temp = main_df[main_df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field_id, data=temp)
        plt.title(f"{numeric_field_id} by {group_field} (top 5 groups)")
        plt.show()

## 6. Conclusion
This notebook demonstrates loading, exploring, and visualizing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.

You can now extend the EDA and build machine learning models or statistical analyses as needed. For further details on field/column definitions and dataset interpretation, please refer to the Croissant schema documentation or supplementary data provided by the dataset authors.